In [8]:
import numpy as np
import pandas as pd
from time import sleep
from pathlib import Path
from bokeh.plotting import figure, show
from scipy.interpolate import make_interp_spline
from scipy.signal import savgol_filter
from bokeh.io import output_notebook, export_png
output_notebook()

Loading BokehJS ...

In [9]:
def normalize(df, norm_data):
	norm = 0
	for l in range(len(norm_data)):
		norm += norm_data[l]

	norm /= len(norm_data)
	df["y"] /= norm
	
	return df

def doThePlot(paths, laser_power, points, color):
	norm_curves = []
	for i in paths:
		rawdf = pd.read_csv(i)
		tmpdf = rawdf.drop(columns="y2")
		tmpdf = tmpdf.rename(columns={"y1": "y"})

		norm_data = tmpdf.iloc[-100:, 1].to_numpy()

		tmpdf = normalize(tmpdf, norm_data)

		norm_curves.append(tmpdf["y"])

	df = pd.DataFrame()

	for c in norm_curves:
		df = pd.concat([df, c], axis=1)

	df = df.mean(axis=1).to_frame(name="y")
	df.insert(0, "x", rawdf["x"])

	if points != 0:
		df["y"] = savgol_filter(df["y"], window_length=points, polyorder=3)

	transmittance_diff_p_v = max(df["y"]) - min(df["y"])
	distance_diff_p_v = (df["x"][np.argmax(df["y"])] - df["x"][np.argmin(df["y"])]) * 10**-2

	n2 = (transmittance_diff_p_v * ((laser_wavelength * 10**-9)**2) * distance_diff_p_v) / (0.406 * 4 * np.pi * (laser_power * 10**-3) * (sample_length * 10**-3) * 1.7)

	print(f"=========== Potencia: {laser_power} mW ===========")
	print(f"n2: {n2} m²/W")
	print(f"ΔTpv: {transmittance_diff_p_v}")
	print(f"Δzpv: {distance_diff_p_v} cm\n")

	source = df.to_dict(orient="list")

	p = figure(width = 1280, height = 720, x_range = (-0.5 , 30.5), y_range=(0, 2.04), x_axis_label="Distancia (cm)", y_axis_label="Transmitancia", tools=["pan", "wheel_zoom", "box_zoom", "reset", "save"])
	p.line(source=source, color=color, legend_label=f"n2: {n2:.5e} m²/W", line_width=2)

	if points != 0:
		p.title = f"Curva promedio a {laser_power} mW - Filtro SG cúbico con {points} puntos"
	else:
		p.title = f"Curva promedio a {laser_power} mW - Sin filtrado"

	p.title.align = "center"
	p.title.text_font_size = "20pt"

	p.xaxis.major_label_text_font_size = "20pt"
	p.xaxis.axis_label_text_font_size = "20pt"

	p.yaxis.major_label_text_font_size = "20pt"
	p.yaxis.axis_label_text_font_size = "20pt"

	p.legend.location = "top_right"
	p.legend.label_text_font_size = "20pt"

	p.y_range.start = min(df["y"]) - 0.05
	p.y_range.end = max(df["y"]) + 0.05

	show(p)

	sleep(1)

	return source, n2


In [ ]:
laser_wavelength = 532
sample_length = 1
points = 70
powers = [10, 20, 60]
colors = ["red", "blue", "green"]

In [11]:
pathsmdo = [
	["Final Measures/Medalla de Oro/[1] 10 mW.csv",
	 "Final Measures/Medalla de Oro/[2] 10 mW.csv",
	 "Final Measures/Medalla de Oro/[3] 10 mW.csv",
	 "Final Measures/Medalla de Oro/[4] 10 mW.csv",
	 "Final Measures/Medalla de Oro/[5] 10 mW.csv"],

	["Final Measures/Medalla de Oro/[1] 20 mW.csv",
	 "Final Measures/Medalla de Oro/[2] 20 mW.csv",
	 "Final Measures/Medalla de Oro/[3] 20 mW.csv",
	 "Final Measures/Medalla de Oro/[4] 20 mW.csv",
	 "Final Measures/Medalla de Oro/[5] 20 mW.csv"],

	["Final Measures/Medalla de Oro/[1] 60 mW.csv",
	 "Final Measures/Medalla de Oro/[2] 60 mW.csv",
	 "Final Measures/Medalla de Oro/[3] 60 mW.csv",
	 "Final Measures/Medalla de Oro/[4] 60 mW.csv",
	 "Final Measures/Medalla de Oro/[5] 60 mW.csv"]
]

sourcesmdo = []
n2mdo = []
for i in range(len(pathsmdo)):
	tmp_source, tmp_n2 = doThePlot(pathsmdo[i], powers[i], points, "red")
	sourcesmdo.append(tmp_source)
	n2mdo.append(tmp_n2)

p = figure(width = 1280, height = 720, x_range = (-0.5 , 30.5), y_range=(0, 2.04), x_axis_label="Distancia (cm)", y_axis_label="Transmitancia", tools=["pan", "wheel_zoom", "box_zoom", "reset", "save"])

for i in range(len(powers)):
	p.line(source=sourcesmdo[i], color=colors[i], legend_label=f"{powers[i]} mW - n2: {n2mdo[i]:.5e} m²/W", line_width=2)

p.title = f"Curvas de transmitancia del aceite 1 (MdO)"

p.title.align = "center"
p.title.text_font_size = "20pt"

p.xaxis.major_label_text_font_size = "20pt"
p.xaxis.axis_label_text_font_size = "20pt"

p.yaxis.major_label_text_font_size = "20pt"
p.yaxis.axis_label_text_font_size = "20pt"

p.legend.location = "top_right"
p.legend.label_text_font_size = "20pt"

p.y_range.start = min(i for s in sourcesmdo for i in s["y"]) - 0.05
p.y_range.end = max(i for s in sourcesmdo for i in s["y"]) + 0.05

show(p)

=========== Potencia: 10 mW ===========
n2: -3.170972123754114e-11 m²/W
ΔTpv: 0.5649701877008434
Δzpv: -0.017199999999999997 cm



=========== Potencia: 20 mW ===========
n2: -3.956027483324785e-11 m²/W
ΔTpv: 1.068132164056061
Δzpv: -0.022700000000000005 cm



=========== Potencia: 60 mW ===========
n2: -3.453954318704148e-11 m²/W
ΔTpv: 1.9186749273920864
Δzpv: -0.0331 cm



In [12]:
pathsexg = [
	["Final Measures/Gallo Clássico/[1] 10 mW.csv",
	 "Final Measures/Gallo Clássico/[2] 10 mW.csv",
	 "Final Measures/Gallo Clássico/[3] 10 mW.csv",
	 "Final Measures/Gallo Clássico/[4] 10 mW.csv",
	 "Final Measures/Gallo Clássico/[5] 10 mW.csv"],

	["Final Measures/Gallo Clássico/[1] 20 mW.csv",
	 "Final Measures/Gallo Clássico/[2] 20 mW.csv",
	 "Final Measures/Gallo Clássico/[3] 20 mW.csv",
	 "Final Measures/Gallo Clássico/[4] 20 mW.csv",
	 "Final Measures/Gallo Clássico/[5] 20 mW.csv"],

	["Final Measures/Gallo Clássico/[1] 60 mW.csv",
	 "Final Measures/Gallo Clássico/[2] 60 mW.csv",
	 "Final Measures/Gallo Clássico/[3] 60 mW.csv",
	 "Final Measures/Gallo Clássico/[4] 60 mW.csv",
	 "Final Measures/Gallo Clássico/[5] 60 mW.csv"]
]

sourcesexg = []
n2exg = []
for i in range(len(pathsexg)):
	tmp_source, tmp_n2 = doThePlot(pathsexg[i], powers[i], points, "red")
	sourcesexg.append(tmp_source)
	n2exg.append(tmp_n2)

p = figure(width = 1280, height = 720, x_range = (-0.5 , 30.5), y_range=(0, 2.04), x_axis_label="Distancia (cm)", y_axis_label="Transmitancia", tools=["pan", "wheel_zoom", "box_zoom", "reset", "save"])

for i in range(len(powers)):
	p.line(source=sourcesexg[i], color=colors[i], legend_label=f"{powers[i]} mW - n2: {n2exg[i]:.5e} m²/W", line_width=2)

p.title = f"Curvas de transmitancia del aceite 2 (EXG)"

p.title.align = "center"
p.title.text_font_size = "20pt"

p.xaxis.major_label_text_font_size = "20pt"
p.xaxis.axis_label_text_font_size = "20pt"

p.yaxis.major_label_text_font_size = "20pt"
p.yaxis.axis_label_text_font_size = "20pt"

p.legend.location = "top_right"
p.legend.label_text_font_size = "20pt"

p.y_range.start = min(i for s in sourcesexg for i in s["y"]) - 0.05
p.y_range.end = max(i for s in sourcesexg for i in s["y"]) + 0.05

show(p)

=========== Potencia: 10 mW ===========
n2: -2.226020251738256e-11 m²/W
ΔTpv: 0.4401077004115912
Δzpv: -0.015499999999999998 cm



=========== Potencia: 20 mW ===========
n2: -2.6089144515002472e-11 m²/W
ΔTpv: 0.8550859724633779
Δzpv: -0.0187 cm



=========== Potencia: 60 mW ===========
n2: -2.940283526875819e-11 m²/W
ΔTpv: 1.761017371421694
Δzpv: -0.030700000000000005 cm



In [13]:
pathsppex = [
	["Final Measures/Gallo PP3/[1] 10 mW.csv",
	 "Final Measures/Gallo PP3/[2] 10 mW.csv",
	 "Final Measures/Gallo PP3/[3] 10 mW.csv",
	 "Final Measures/Gallo PP3/[4] 10 mW.csv",
	 "Final Measures/Gallo PP3/[5] 10 mW.csv"],

	["Final Measures/Gallo PP3/[1] 20 mW.csv",
	 "Final Measures/Gallo PP3/[2] 20 mW.csv"],

	["Final Measures/Gallo PP3/[1] 60 mW.csv",
	 "Final Measures/Gallo PP3/[2] 60 mW.csv"]
]

sourcesppex = []
n2ppex = []
for i in range(len(pathsppex)):
	tmp_source, tmp_n2 = doThePlot(pathsppex[i], powers[i], points, "red")
	sourcesppex.append(tmp_source)
	n2ppex.append(tmp_n2)

p = figure(width = 1280, height = 720, x_range = (-0.5 , 30.5), y_range=(0, 2.04), x_axis_label="Distancia (cm)", y_axis_label="Transmitancia", tools=["pan", "wheel_zoom", "box_zoom", "reset", "save"])

for i in range(len(powers)):
	p.line(source=sourcesppex[i], color=colors[i], legend_label=f"{powers[i]} mW - n2: {n2ppex[i]:.5e} m²/W", line_width=2)

p.title = f"Curvas de transmitancia del aceite 3 (PPEX)"

p.title.align = "center"
p.title.text_font_size = "20pt"

p.xaxis.major_label_text_font_size = "20pt"
p.xaxis.axis_label_text_font_size = "20pt"

p.yaxis.major_label_text_font_size = "20pt"
p.yaxis.axis_label_text_font_size = "20pt"

p.legend.location = "top_right"
p.legend.label_text_font_size = "20pt"

p.y_range.start = min(i for s in sourcesppex for i in s["y"]) - 0.05
p.y_range.end = max(i for s in sourcesppex for i in s["y"]) + 0.05

show(p)

=========== Potencia: 10 mW ===========
n2: -4.740381689585943e-10 m²/W
ΔTpv: 2.2487560683837096
Δzpv: -0.0646 cm



=========== Potencia: 20 mW ===========
n2: 4.0123486301407504e-10 m²/W
ΔTpv: 1.1077384669327988
Δzpv: 0.222 cm



=========== Potencia: 60 mW ===========
n2: 1.3509678438759433e-10 m²/W
ΔTpv: 1.0952537945759635
Δzpv: 0.2268 cm

